# Spatial Analysis

Identifies where chronicity clusters: Getis-Ord Gi* hotspots, Local Moran's I clusters and outliers, severity-weighted kernel density and hexagonal (H3) aggregation, using the composite index and the geolocated events.

Outputs: `hotspots_*`, `local_moran_*`, `kde_*` and `hexagonos_*` files in `data/processed/`.

## 0. Setup

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Numerical operations
import numpy as np
# Tabular data handling
import pandas as pd
# Geospatial data handling
import geopandas as gpd
# Queen contiguity weights and the generic weights class
from libpysal.weights import Queen, W
# Getis-Ord Gi* and Local Moran's I statistics
from esda import G_Local, Moran_Local
# H3 hexagonal grid library
import h3
# Gaussian kernel density estimator
from scipy.stats import gaussian_kde

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)
# Create a logger that writes to the logs folder
logger = get_logger("step9_spatial_analysis", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("análisis espacial iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Spatial analysis settings from the config
SPA_CFG = config["spatial_analysis"]
# Variables to analyze (composite index and its components)
VARIABLES = SPA_CFG["variables"]
# Number of permutations for significance tests
PERMUTATIONS = SPA_CFG["significance"]["permutations"]
# P-value thresholds for 99%, 95% and 90% confidence
P_THRESHOLDS = SPA_CFG["significance"]["p_thresholds"]
# H3 resolution of the hexagons
H3_RES = SPA_CFG["hexagons"]["h3_resolution"]
# Kernel bandwidth method
KDE_BW = SPA_CFG["kde"]["bw_method"]
# Kernel grid resolution in kilometres
KDE_GRID_KM = SPA_CFG["kde"]["grid_resolution_km"]
# Metric coordinate system used for distances
CRS_METRIC = SPA_CFG["kde"]["crs_metric"]

# Fixed random seed for reproducibility
RNG_SEED = 42

# Log the variables to analyze
logger.info("análisis espacial iniciada. Variables a analizar: %s", VARIABLES)

## 1. Inputs: canton geometry, composite index and geolocated events

In [ ]:
# Read canton polygons
gdf_canton = gpd.read_file(processed_dir / "adm2_canton.geojson")
# Make canton codes strings
gdf_canton["canton_code"] = gdf_canton["canton_code"].astype(str)
# Log the number of polygons and the CRS
logger.info("Cantones cargados: %d polígonos, CRS=%s", len(gdf_canton), gdf_canton.crs)

# Read the composite index at canton level
efci = pd.read_csv(processed_dir / "efci_composite_canton.csv", dtype={"canton_code": str})
# Keep the combined hazard view
efci_todos = efci[efci["hazard_type"] == "TODOS"].copy()
# Check all 221 cantons are present
assert len(efci_todos) == 221, f"se esperaban 221 cantones en la vista TODOS, hay {len(efci_todos)}"

# Join the index values to the canton polygons
gdf = gdf_canton.merge(
    efci_todos[["canton_code"] + VARIABLES],
    on="canton_code", how="left",
)
# Check no canton lacks a value
assert gdf[VARIABLES].isna().sum().sum() == 0, "hay cantones sin valor de EFCI o algún sub-índice — revisar el merge"
# Reset the index so positions match the weights matrix
gdf = gdf.reset_index(drop=True)
# Log the size of the working table
logger.info("Tabla espacial de trabajo: %d cantones x %d variables", len(gdf), len(VARIABLES))

# Read the geolocated events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Log the number of events
logger.info("Eventos geolocalizados cargados: %d", len(he))

## 2. Spatial weights: Queen contiguity with KNN fallback

In [ ]:
# Queen contiguity: cantons sharing a border or a corner are neighbors
w_queen = Queen.from_dataframe(gdf, use_index=True)
# Log the coverage check
logger.info(
    "Queen: %d vecindades, %d cantones sin vecino (islas): %s",
    w_queen.n, len(w_queen.islands), w_queen.islands,
)

# Check for islands (cantons without neighbors)
if w_queen.islands:
    # Names of the island cantons
    islas_nombres = gdf.loc[w_queen.islands, "canton_name_official"].tolist()
    # Log the island cantons
    logger.info("Cantones sin vecino por contigüidad: %s", islas_nombres)

In [ ]:
# Reproject to a metric CRS to measure distances
gdf_m = gdf.to_crs(CRS_METRIC)
# Canton centroids
centroids = gdf_m.geometry.centroid

# Neighbor lists from the Queen weights
neighbors = {i: list(w_queen.neighbors[i]) for i in range(len(gdf))}
# Binary weights for each neighbor
weights_bin = {i: [1.0] * len(neighbors[i]) for i in neighbors}

# Loop over the island cantons
for idx in w_queen.islands:
    # Distances from the island to every other centroid
    dists = centroids.distance(centroids.iloc[idx])
    # Sort by distance, excluding the island itself
    dists = dists.drop(index=idx).sort_values()
    # Index of the nearest canton
    vecino_idx = int(dists.index[0])
    # Distance to the nearest canton in kilometres
    dist_km = dists.iloc[0] / 1000
    # Log a warning so the correction is auditable
    logger.warning(
        "Respaldo KNN aplicado: '%s' no tiene vecino por contigüidad -> "
        "conectado a '%s' (vecino más cercano, %.1f km)",
        gdf.loc[idx, "canton_name_official"], gdf.loc[vecino_idx, "canton_name_official"], dist_km,
    )
    # Add the nearest canton as a neighbor if missing
    if vecino_idx not in neighbors[idx]:
        # Add the neighbor
        neighbors[idx].append(vecino_idx)
        # Give it a binary weight
        weights_bin[idx].append(1.0)
    # Make the link symmetric
    if idx not in neighbors[vecino_idx]:
        # Add the island as a neighbor of the nearest canton
        neighbors[vecino_idx].append(idx)
        # Give it a binary weight
        weights_bin[vecino_idx].append(1.0)

# Combined weights matrix (Queen plus nearest neighbor)
w_combined = W(neighbors, weights_bin)
# Use binary weights
w_combined.transform = "b"
# Check no islands remain
assert len(w_combined.islands) == 0, "quedaron islas sin resolver tras el respaldo KNN"
# Log the size of the combined weights
logger.info("Matriz de pesos combinada (Queen+KNN): 0 islas, %d cantones conectados", w_combined.n)

# Row-standardized copy of the weights for Local Moran's I
w_rowstd = W(neighbors, weights_bin)
# Use row-standardized weights
w_rowstd.transform = "r"

In [ ]:
# Loop over the island cantons
for idx in w_queen.islands:
    # Names of the neighbors assigned to the island
    vecinos_nombres = [gdf.loc[j, "canton_name_official"] for j in neighbors[idx]]
    # Print each island with its neighbors
    print(f"{gdf.loc[idx, 'canton_name_official']:25s} -> conectado a: {vecinos_nombres}")

## 3. Getis-Ord Gi*: chronicity hotspots and coldspots

In [ ]:
# Classify a Gi* result as hotspot or coldspot at 99%, 95% or 90%
def clasificar_gi(z, p):
    """Classify a Getis-Ord Gi* result as a hotspot or coldspot at 99%, 95% or 90% confidence.
    
    Args:
        z: Gi* z-score.
        p: Pseudo p-value from the permutation test.
    
    Returns:
        'No significativo', 'Punto Caliente <level>' or 'Punto Frío <level>'.
    """
    # Not significant above the 90% threshold
    if p > P_THRESHOLDS[2]:
        # Label for non-significant cantons
        return "No significativo"
    # Confidence level of the result
    nivel = "99%" if p <= P_THRESHOLDS[0] else ("95%" if p <= P_THRESHOLDS[1] else "90%")
    # Hotspot when the z-score is positive, coldspot otherwise
    return f"Punto Caliente {nivel}" if z > 0 else f"Punto Frío {nivel}"


# Dictionary that collects the Gi* results
gi_resultados = {"canton_code": gdf["canton_code"], "canton_name_official": gdf["canton_name_official"]}

# Loop over the variables
for var in VARIABLES:
    # Values of the variable
    y = gdf[var].values
    # Getis-Ord Gi* with permutation-based significance
    gi = G_Local(y, w_combined, transform="B", star=True, permutations=PERMUTATIONS, seed=RNG_SEED)
    # Store the z-scores
    gi_resultados[f"gi_z_{var}"] = gi.Zs
    # Store the pseudo p-values
    gi_resultados[f"gi_p_{var}"] = gi.p_sim
    # Store the hotspot or coldspot class
    gi_resultados[f"gi_clase_{var}"] = [clasificar_gi(z, p) for z, p in zip(gi.Zs, gi.p_sim)]
    # Count hotspots
    n_hot = sum(1 for c in gi_resultados[f"gi_clase_{var}"] if "Caliente" in c)
    # Count coldspots
    n_cold = sum(1 for c in gi_resultados[f"gi_clase_{var}"] if "Frío" in c)
    # Log the counts
    logger.info("Gi* (%s): %d puntos calientes, %d puntos fríos, de %d cantones", var, n_hot, n_cold, len(y))

# Build the Gi* table
gi_df = pd.DataFrame(gi_resultados)
# Preview the table
gi_df.head()

## 4. Local Moran's I: clusters and spatial outliers

In [ ]:
# Labels of the four Moran quadrants
QUADRANT_LABELS = {1: "Alto-Alto (HH)", 2: "Bajo-Alto (LH)", 3: "Bajo-Bajo (LL)", 4: "Alto-Bajo (HL)"}

# Dictionary that collects the Local Moran results
moran_resultados = {"canton_code": gdf["canton_code"], "canton_name_official": gdf["canton_name_official"]}

# Loop over the variables
for var in VARIABLES:
    # Values of the variable
    y = gdf[var].values
    # Local Moran's I with permutation-based significance
    ml = Moran_Local(y, w_rowstd, permutations=PERMUTATIONS, seed=RNG_SEED)
    # Mark significant cantons at the 95% threshold
    sig = ml.p_sim <= P_THRESHOLDS[1]
    # Store the local statistic
    moran_resultados[f"moran_i_{var}"] = ml.Is
    # Store the pseudo p-values
    moran_resultados[f"moran_p_{var}"] = ml.p_sim
    # Store the quadrant label for significant cantons
    moran_resultados[f"moran_cuadrante_{var}"] = [
        QUADRANT_LABELS[q] if s else "No significativo" for q, s in zip(ml.q, sig)
    ]
    # Count high-high clusters
    n_hh = sum(1 for c in moran_resultados[f"moran_cuadrante_{var}"] if "HH" in c)
    # Count low-low clusters
    n_ll = sum(1 for c in moran_resultados[f"moran_cuadrante_{var}"] if "LL" in c)
    # Log the cluster counts
    logger.info("Local Moran's I (%s): %d clusters HH, %d clusters LL (95%%), de %d cantones", var, n_hh, n_ll, len(y))

# Build the Local Moran table
moran_df = pd.DataFrame(moran_resultados)
# Preview the table
moran_df.head()

## 5. Validation of sections 3-4

In [ ]:
# Loop over both result tables
for df_check, nombre in [(gi_df, "Gi*"), (moran_df, "Local Moran's I")]:
    # Check all 221 cantons are present
    assert len(df_check) == 221, f"{nombre}: se esperaban 221 cantones, hay {len(df_check)}"
    # Numeric result columns
    cols_numericas = [c for c in df_check.columns if c.startswith(("gi_z_", "gi_p_", "moran_i_", "moran_p_"))]
    # Check there are no missing values
    assert df_check[cols_numericas].isna().sum().sum() == 0, f"{nombre}: hay valores nulos"
    # Check all values are finite
    assert np.isfinite(df_check[cols_numericas].values).all(), f"{nombre}: hay valores infinitos"

# Gi* class of Quito as a sanity check
quito_gi = gi_df.loc[gi_df["canton_name_official"].str.contains("Quito", na=False), "gi_clase_efci"].values
# Local Moran class of Quito as a sanity check
quito_moran = moran_df.loc[moran_df["canton_name_official"].str.contains("Quito", na=False), "moran_cuadrante_efci"].values
# Log the Quito result
logger.info("Verificación de sentido común — Quito: Gi*(efci)=%s, Moran(efci)=%s", quito_gi, quito_moran)
# Print the Gi* result for Quito
print("Gi* Quito (efci):", quito_gi)
# Print the Local Moran result for Quito
print("Moran's I Quito (efci):", quito_moran)

## 6. Kernel density weighted by event severity

In [ ]:
# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Pillar weights from the severity configuration
PESOS = SEV_CFG["pillar_weights"]

# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"])
    + W_INJURED * he["injured"]
)
# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]
# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]
# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]
# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)

# Names of the pillar score columns
pilares = ["score_poblacion", "score_vivienda", "score_agricultura", "score_ganaderia", "score_infraestructura"]
# Loop over the pillars
for col in pilares:
    # Minimum and maximum of the pillar score
    minimo, maximo = he[col].min(), he[col].max()
    # Range of the pillar score
    rango = maximo - minimo
    # Min-max normalization of the pillar
    he[col.replace("score_", "norm_")] = (he[col] - minimo) / rango if rango > 0 else 0.0

# Event severity score: weighted sum of the normalized pillars
he["event_severity_score"] = (
    PESOS["poblacion"] * he["norm_poblacion"]
    + PESOS["vivienda"] * he["norm_vivienda"]
    + PESOS["agricultura"] * he["norm_agricultura"]
    + PESOS["ganaderia"] * he["norm_ganaderia"]
    + PESOS["infraestructura"] * he["norm_infraestructura"]
)
# Check the event severity score is between 0 and 1
assert he["event_severity_score"].between(0, 1).all(), "event_severity_score fuera de [0, 1]"
# Log the coverage check
logger.info(
    "event_severity_score calculado: media=%.4f, máximo=%.4f (evento %s)",
    he["event_severity_score"].mean(), he["event_severity_score"].max(),
    he.loc[he["event_severity_score"].idxmax(), "event_id"],
)

In [ ]:
# Reproject events to the metric CRS
he_m = he.to_crs(CRS_METRIC)
# X coordinates in metres
xs = he_m.geometry.x.values
# Y coordinates in metres
ys = he_m.geometry.y.values
# Kernel weights from the event severity, with a small constant so no event has zero weight
pesos_kde = he["event_severity_score"].values + 0.01

# Fit a severity-weighted Gaussian kernel density
kde = gaussian_kde(np.vstack([xs, ys]), weights=pesos_kde, bw_method=KDE_BW)
# Log the kernel bandwidth
logger.info("KDE ajustado: bandwidth factor (Silverman)=%.1f m", kde.factor * np.std(np.hstack([xs, ys])))

# Name of the province column
provincia_col = "province" if "province" in he_m.columns else "province_name_official"
# Exclude the Galapagos Islands from the continental grid
continental = he_m[~he_m[provincia_col].str.upper().str.contains("GALAPAGOS|GALÁPAGOS", na=False)]
# Bounding box of continental events
minx, miny, maxx, maxy = continental.total_bounds
# Margin around the bounding box, in metres
margen = 20_000
# Grid step in metres
paso = KDE_GRID_KM * 1_000
# Regular grid of x and y coordinates
xx, yy = np.meshgrid(
    np.arange(minx - margen, maxx + margen, paso),
    np.arange(miny - margen, maxy + margen, paso),
)
# Flatten the grid into a 2 x N array of points
grid_xy = np.vstack([xx.ravel(), yy.ravel()])
# Evaluate the density at every grid point
densidad = kde(grid_xy)
# Normalize the density to the 0-1 range
densidad_norm = (densidad - densidad.min()) / (densidad.max() - densidad.min())

# Build the density grid as a GeoDataFrame
grid_gdf = gpd.GeoDataFrame(
    {"densidad_kde": densidad, "densidad_kde_norm": densidad_norm},
    geometry=gpd.points_from_xy(grid_xy[0], grid_xy[1]),
    crs=CRS_METRIC,
).to_crs(4326)
# Log the number of grid points
logger.info("Grilla KDE evaluada: %d puntos (paso=%d km)", len(grid_gdf), KDE_GRID_KM)

## 7. Hexagonal aggregation (H3 resolution 5, about 10 km)

In [ ]:
# Assign each event to its H3 hexagon
he["h3_cell"] = he.apply(lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1)

# Aggregate events per hexagon (count and severity statistics)
hex_agg = he.groupby("h3_cell").agg(
    n_eventos=("event_id", "count"),
    severity_mean=("event_severity_score", "mean"),
    severity_max=("event_severity_score", "max"),
    severity_sum=("event_severity_score", "sum"),
    hazard_type_dominante=("hazard_type", lambda s: s.value_counts().idxmax()),
).reset_index()

# Shapely geometry helpers
import shapely.geometry as shp_geom
# Build each hexagon's polygon from its H3 cell
hex_agg["geometry"] = hex_agg["h3_cell"].apply(
    lambda c: shp_geom.Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])
)
# Create the hexagon GeoDataFrame
hex_gdf = gpd.GeoDataFrame(hex_agg, geometry="geometry", crs=4326)

# Log the coverage check
logger.info(
    "Agregación hexagonal (res=%d, ~10km): %d celdas con >=1 evento, máximo %d eventos en una celda",
    H3_RES, len(hex_gdf), hex_gdf["n_eventos"].max(),
)
# Show the ten hexagons with the most events
hex_gdf.sort_values("n_eventos", ascending=False).head(10)[["h3_cell", "n_eventos", "severity_mean", "hazard_type_dominante"]]

## 8. Final validation and export

In [ ]:
# Check hexagon counts add up to the number of events
assert hex_gdf["n_eventos"].sum() == len(he), "la suma de eventos por hexágono no coincide con el total de eventos"
# Check the density grid has no missing values
assert grid_gdf["densidad_kde"].notna().all(), "hay valores nulos en la grilla KDE"
# Check all hexagon geometries are valid
assert (hex_gdf.geometry.is_valid).all(), "hay geometrías de hexágono inválidas"

# Create the output folder if needed
out_dir = ensure_dir(processed_dir)

# Export Gi* hotspots as Parquet
gi_df.to_parquet(out_dir / "hotspots_gi_star_canton.parquet", index=False)
# Export Gi* hotspots as CSV
gi_df.to_csv(out_dir / "hotspots_gi_star_canton.csv", index=False)

# Export Local Moran results as Parquet
moran_df.to_parquet(out_dir / "local_moran_canton.parquet", index=False)
# Export Local Moran results as CSV
moran_df.to_csv(out_dir / "local_moran_canton.csv", index=False)

# Export the density grid as GeoJSON
grid_gdf.to_file(out_dir / "kde_severidad_grid.geojson", driver="GeoJSON")
# Export the density grid as CSV with coordinates
grid_gdf.drop(columns="geometry").assign(
    longitude=grid_gdf.geometry.x, latitude=grid_gdf.geometry.y
).to_csv(out_dir / "kde_severidad_grid.csv", index=False)

# Export hexagons as GeoJSON
hex_gdf.to_file(out_dir / "hexagonos_h3_res5.geojson", driver="GeoJSON")
# Export hexagons as CSV
hex_gdf.drop(columns="geometry").to_csv(out_dir / "hexagonos_h3_res5.csv", index=False)

# Log that all outputs were exported
logger.info("análisis espacial: todas las salidas exportadas a %s", out_dir)
# Print the Gi* export
print("Exportado: hotspots_gi_star_canton.{parquet,csv}")
# Print the Local Moran export
print("Exportado: local_moran_canton.{parquet,csv}")
# Print the density export
print("Exportado: kde_severidad_grid.{geojson,csv}")
# Print the hexagon export
print("Exportado: hexagonos_h3_res5.{geojson,csv}")